<a href="https://colab.research.google.com/github/Syed8855/FlyRank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [7]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

features_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_trailing_month,
        SUM(gsc_impressions) AS impressions_trailing_month,
        AVG(gsc_avg_position) AS avg_position_trailing_month,
        SUM(COALESCE(ga4_sessions, 0)) AS sessions_trailing_month,
        SUM(COALESCE(scroll_events, 0)) AS scroll_events_trailing_month,
        SUM(COALESCE(sessions_organic, 0)) AS organic_sessions_trailing_month,
        AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE NULL END) AS avg_ctr_trailing_month,
        COUNT(*) AS days_with_any_row
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

features_df['avg_ctr_trailing_month'] = features_df['avg_ctr_trailing_month'].fillna(0)

features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,clicks_trailing_month,impressions_trailing_month,avg_position_trailing_month,sessions_trailing_month,scroll_events_trailing_month,organic_sessions_trailing_month,avg_ctr_trailing_month,days_with_any_row
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,7.0,6523.0,7.209549,1.0,0.0,1.0,0.001090,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0.0,453.0,2.987198,0.0,0.0,0.0,0.000000,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.0,5630.0,6.724039,3.0,0.0,6.0,0.001244,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,13.0,4944.0,7.244844,2.0,0.0,1.0,0.003137,31
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,1.0,429.0,4.209227,2.0,1.0,0.0,0.002933,31


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

- clicks_trailing_month — GSC clicks summed over March. No missing values (row exists due to
  gsc_data_available filter). Available before decision moment: yes, purely historical/trailing.
- impressions_trailing_month — GSC impressions summed over March. Same availability as above.
- avg_position_trailing_month — average SERP position over the month. Available before decision
  moment: yes.
- sessions_trailing_month — GA4 sessions; filled with 0 via COALESCE since ga4_data_available can
  be independently FALSE even when gsc_data_available is TRUE. Available before decision moment: yes.
- scroll_events_trailing_month — same missing-value handling as sessions_trailing_month.
- organic_sessions_trailing_month — organic-channel sessions only; same COALESCE handling.
- avg_ctr_trailing_month — engineered ratio (clicks/impressions), undefined when impressions=0,
  filled with 0. Available before decision moment: yes, but CONFIRMED as a mild leak risk in Section 3
  (overlaps with clicks_trailing_month's signal) — flagged for exclusion.
- days_with_any_row — count of days this content had any tracked row in the month; a coverage/volume
  proxy. Available before decision moment: yes. Tested clean in the leakage hunt.
- No categorical features used — client_hash_id/content_hash_id are identifiers only, never fed
  to the model directly.
  

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [8]:
from sklearn.linear_model import LogisticRegression

features_df['is_declining_label'] = (
    features_df['clicks_trailing_month'].rank(pct=True) <= 0.5
).astype(int)

honest_cols = ['impressions_trailing_month', 'avg_position_trailing_month',
               'sessions_trailing_month', 'organic_sessions_trailing_month']
y = features_df['is_declining_label']

# Baseline honest score
X_honest = features_df[honest_cols].fillna(0)
honest_score = LogisticRegression().fit(X_honest, y).score(X_honest, y)
print("Honest baseline:", honest_score)

# Attack candidates — test each suspicious column individually
suspects = {
    'raw_clicks_copy': features_df['clicks_trailing_month'],
    'ctr_derived_from_clicks': features_df['avg_ctr_trailing_month'],  # partially derived from clicks
    'days_with_any_row': features_df['days_with_any_row'],  # could correlate with label via reporting bias
}

for name, col in suspects.items():
    X_test = X_honest.copy()
    X_test[name] = col.fillna(0)
    score = LogisticRegression(max_iter = 1000).fit(X_test, y).score(X_test, y)
    print(f"{name}: score={score:.4f} (delta={score - honest_score:+.4f})")

Honest baseline: 0.8905724858264776
raw_clicks_copy: score=1.0000 (delta=+0.1094)
ctr_derived_from_clicks: score=0.9022 (delta=+0.0116)
days_with_any_row: score=0.8909 (delta=+0.0003)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- raw_clicks_copy (clicks_trailing_month duplicated as a feature) — CONFIRMED LEAK: score jumped
  to 1.0000 (delta +0.1094). Direct copy of the same signal the label is derived from; excluded entirely.

- avg_ctr_trailing_month — MIXED, excluded out of caution: delta +0.0116, since its numerator
  overlaps with clicks_trailing_month, the label-defining column. Not as severe as a direct copy,
  but not clean either — excluded from the honest feature set.

- gsc_sum_position — excluded as redundant: no unique signal beyond avg_position_trailing_month,
  which already captures the same information.

- trend_direction / trend_pct (not present in this table, relevant for future capstone rollups) —
  excluded categorically going forward, since they would be computed from the same outcome the
  label measures.

- client_hash_id / content_hash_id — excluded as model inputs: these are identifiers, not signal;
  using them directly risks the model memorizing per-entity behavior rather than learning
  generalizable patterns.

- days_with_any_row — NOT excluded: tested clean in the leakage hunt (delta +0.0003), kept as a
  legitimate coverage/volume feature.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.